Collecting all the information, cleaning and saving it locally

IMPORTS

In [1]:
import requests
import json
import time
import os
from tqdm.notebook import tqdm

# Create folders if they don't exist
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data", exist_ok=True)

print("✅ Imports done, folders ready")

✅ Imports done, folders ready


API FETCHER FUNCTION

In [2]:
BASE_URL = "https://api.myscheme.gov.in/search/v6/schemes"

def fetch_schemes(from_index=0, size=50):
    params = {
        "lang": "en",
        "q": "",
        "keyword": "",
        "sort": "",
        "from": from_index,
        "size": size,
    }
    headers = {
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9,en-IN;q=0.8",
        "Origin": "https://www.myscheme.gov.in",
        "Referer": "https://www.myscheme.gov.in/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0",
        "X-Api-Key": "tYTy5eEhlu9rFjyxuCr7ra7ACp4dv1RH8gWuHTDc",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-site",
    }
    try:
        response = requests.get(BASE_URL, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"❌ Error at index {from_index}: {e}")
        return None

# Test with first 2 schemes
test = fetch_schemes(from_index=0, size=2)
if test:
    print("✅ API is reachable")
    print(f"Sample keys: {list(test.keys())}")
else:
    print("❌ API call failed")

✅ API is reachable
Sample keys: ['status', 'statusCode', 'errorDescription', 'error', 'data']


FETCHING ALL THE SCHEMES

In [3]:
all_schemes = []
from_index = 0
size = 50

print("🔄 Fetching all the schemes...")

while True:
    data = fetch_schemes(from_index=from_index, size=size)
    
    if not data or data.get("status") != "Success":
        print(f"⚠️ Bad response at index {from_index}, stopping.")
        break
    
    try:
        hits = data["data"]["hits"]["items"]
    except (KeyError, TypeError):
        print(f"⚠️ Unexpected structure at index {from_index}")
        break
    
    if not hits:
        print(f"✅ No more schemes at index {from_index}. Done!")
        break
    
    all_schemes.extend(hits)
    print(f"📦 Fetched {len(hits)} | Total: {len(all_schemes)}")
    
    from_index += size
    time.sleep(0.5)

print(f"\n✅ Total schemes fetched: {len(all_schemes)}")

🔄 Fetching all the schemes...
📦 Fetched 50 | Total: 50
📦 Fetched 50 | Total: 100
📦 Fetched 50 | Total: 150
📦 Fetched 50 | Total: 200
📦 Fetched 50 | Total: 250
📦 Fetched 50 | Total: 300
📦 Fetched 50 | Total: 350
📦 Fetched 50 | Total: 400
📦 Fetched 50 | Total: 450
📦 Fetched 50 | Total: 500
📦 Fetched 50 | Total: 550
📦 Fetched 50 | Total: 600
📦 Fetched 50 | Total: 650
📦 Fetched 50 | Total: 700
📦 Fetched 50 | Total: 750
📦 Fetched 50 | Total: 800
📦 Fetched 50 | Total: 850
📦 Fetched 50 | Total: 900
📦 Fetched 50 | Total: 950
📦 Fetched 50 | Total: 1000
📦 Fetched 50 | Total: 1050
📦 Fetched 50 | Total: 1100
📦 Fetched 50 | Total: 1150
📦 Fetched 50 | Total: 1200
📦 Fetched 50 | Total: 1250
📦 Fetched 50 | Total: 1300
📦 Fetched 50 | Total: 1350
📦 Fetched 50 | Total: 1400
📦 Fetched 50 | Total: 1450
📦 Fetched 50 | Total: 1500
📦 Fetched 50 | Total: 1550
📦 Fetched 50 | Total: 1600
📦 Fetched 50 | Total: 1650
📦 Fetched 50 | Total: 1700
📦 Fetched 50 | Total: 1750
📦 Fetched 50 | Total: 1800
📦 Fetched 50 | Tot

SAVE RAW DATA

In [7]:
# Save the raw untouched response as backup
raw_path = "../data/raw/schemes_raw.json"

with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(all_schemes, f, indent=2, ensure_ascii=False)

print(f"✅ Raw data saved to {raw_path}")
print(f"📊 Total schemes in file: {len(all_schemes)}")

# Preview one scheme to understand the structure
print("\n🔍 Sample scheme keys:")
if all_schemes:
    sample = all_schemes[0].get("_source", {})
    for key in list(sample.keys())[:15]:
        print(f"  - {key}: {str(sample[key])[:80]}")

✅ Raw data saved to ../data/raw/schemes_raw.json
📊 Total schemes in file: 4718

🔍 Sample scheme keys:


CLEANING AND STRUCTURING THE DATA

In [8]:
def clean_scheme(item):
    src = item.get("fields", {})
    
    title       = src.get("schemeName", "").strip()
    ministry    = src.get("nodalMinistryName", "").strip()
    state       = ", ".join(src.get("beneficiaryState", ["Central"]))
    description = src.get("briefDescription", "").strip()
    eligibility = src.get("eligibility", "").strip()
    benefits    = src.get("benefitTypes", "")
    how_to      = src.get("applicationProcess", "").strip()
    slug        = src.get("slug", "")
    tags        = src.get("tags", [])
    level       = src.get("level", "")
    category    = src.get("schemeCategory", [])

    if not title or not description:
        return None

    full_text = f"""
Scheme Name: {title}
Ministry: {ministry}
State: {state}
Level: {level}
Category: {', '.join(category) if isinstance(category, list) else category}
Description: {description}
Eligibility: {eligibility}
Benefits: {benefits}
Tags: {', '.join(tags) if isinstance(tags, list) else tags}
    """.strip()

    return {
        "id": item.get("id", ""),
        "title": title,
        "ministry": ministry,
        "state": state,
        "level": level,
        "category": category,
        "description": description,
        "eligibility": eligibility,
        "benefits": benefits,
        "how_to_apply": how_to,
        "tags": tags,
        "url": f"https://www.myscheme.gov.in/schemes/{slug}",
        "full_text": full_text
    }

cleaned = []
skipped = 0

for item in all_schemes:
    result = clean_scheme(item)
    if result:
        cleaned.append(result)
    else:
        skipped += 1

print(f"✅ Cleaned schemes: {len(cleaned)}")
print(f"⚠️  Skipped (missing data): {skipped}")

✅ Cleaned schemes: 4718
⚠️  Skipped (missing data): 0


SAVING CLEAN DATA AND VERIFY

In [9]:
# Save cleaned data
clean_path = "../data/schemes.json"

with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

print(f"✅ Clean data saved to {clean_path}")
print(f"📊 Total usable schemes: {len(cleaned)}")

# Final verification
print("\n🔍 Sample cleaned scheme:")
print("-" * 50)
sample = cleaned[0]
print(f"Title      : {sample['title']}")
print(f"Ministry   : {sample['ministry']}")
print(f"State      : {sample['state']}")
print(f"URL        : {sample['url']}")
print(f"\nFull Text Preview:\n{sample['full_text'][:300]}...")

✅ Clean data saved to ../data/schemes.json
📊 Total usable schemes: 4718

🔍 Sample cleaned scheme:
--------------------------------------------------
Title      : Stand-Up India
Ministry   : Ministry Of Finance
State      : All
URL        : https://www.myscheme.gov.in/schemes/sui

Full Text Preview:
Scheme Name: Stand-Up India
Ministry: Ministry Of Finance
State: All
Level: Central
Category: Business & Entrepreneurship, Banking,Financial Services and Insurance, Social welfare & Empowerment
Description: A scheme by Ministry of Finance for financing SC/ST and  Women Entrepreneurs by facilitating ...
